# Ranked-storm CSV and JSON consistency check

Run this notebook after StormHub finishes building a storm catalog. It compares each row in `ranked-storms.csv` with the JSON file in the matching numbered event folder.

The check covers the storm start time and the mean, minimum, and maximum precipitation values. The Excel report contains a run summary, mismatch counts, mismatch details, all folder comparisons, and CSV rows that do not have numbered folders.

## Workflow position

1. Select the transposition domain with `mtools_01_domain_selection.ipynb`.
2. Run StormHub.
3. Run this notebook to check the ranked CSV and event JSON files.
4. Continue with `mtools_03_storm_catalog_maps.ipynb`.

## Imports

In [ ]:
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

## File paths and settings

Set `EVENTS_DIR` to the StormHub duration folder that contains `ranked-storms.csv` and the numbered event folders. Set `OUTPUT_DIR` to the report folder.

In [ ]:
EVENTS_DIR = Path('/workspaces/meteorology-tools/inputs/storm_catalog/72hr-events')
OUTPUT_DIR = Path('/workspaces/meteorology-tools/outputs/02_storm_catalog_consistency')

CSV_NAME = 'ranked-storms.csv'
EXCEL_NAME = 'ranked_storm_json_consistency.xlsx'
NUMERIC_TOLERANCE = 1e-6
DISPLAY_ROWS = 25

CSV_PATH = EVENTS_DIR / CSV_NAME
EXCEL_PATH = OUTPUT_DIR / EXCEL_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert EVENTS_DIR.is_dir(), EVENTS_DIR
assert CSV_PATH.is_file(), CSV_PATH
assert NUMERIC_TOLERANCE >= 0, 'NUMERIC_TOLERANCE must be zero or greater.'

print('Events folder:', EVENTS_DIR)
print('Ranked CSV:', CSV_PATH)
print('Excel report:', EXCEL_PATH)

## Comparison rules

- A numbered folder represents the same rank in `ranked-storms.csv`.
- The preferred JSON filename matches the folder number, such as `12/12.json`.
- If that file is absent and the folder has one JSON file, the notebook uses that file and records a note.
- Dates are converted to a common UTC timestamp before comparison.
- Mean, minimum, and maximum values use the absolute tolerance set above.
- Cross-rank columns show whether the JSON values match another CSV rank.

In [ ]:
@dataclass(frozen=True)
class FieldCheck:
    name: str
    csv_value: str
    json_value: str
    csv_normalized: str
    json_normalized: str
    difference: str
    match: bool


def clean_header(name: str) -> str:
    return name.strip().lower().replace(' ', '_')


def parse_datetime(value: Any) -> datetime | None:
    if value in (None, ''):
        return None
    text = str(value).strip()
    if not text:
        return None

    candidates = [text]
    if text.endswith('Z'):
        candidates.append(text[:-1] + '+00:00')
    if 'T' in text and len(text) == 13:
        candidates.append(text + ':00:00')
    if 'T' in text and len(text) == 16:
        candidates.append(text + ':00')

    for candidate in candidates:
        try:
            parsed = datetime.fromisoformat(candidate)
        except ValueError:
            continue
        if parsed.tzinfo is not None:
            parsed = parsed.astimezone(timezone.utc).replace(tzinfo=None)
        return parsed

    for date_format in ('%Y-%m-%d', '%Y%m%d'):
        try:
            return datetime.strptime(text, date_format)
        except ValueError:
            pass
    return None


def format_datetime(value: datetime | None) -> str:
    return '' if value is None else value.strftime('%Y-%m-%dT%H:%M:%S')


def parse_number(value: Any) -> float | None:
    if value in (None, ''):
        return None
    try:
        return float(str(value).strip())
    except (TypeError, ValueError):
        return None


def number_key(value: Any) -> str:
    number = parse_number(value)
    return '' if number is None else f'{number:.6f}'


def storm_value_key(start: Any, mean: Any, minimum: Any, maximum: Any) -> tuple[str, str, str, str]:
    return (
        format_datetime(parse_datetime(start)),
        number_key(mean),
        number_key(minimum),
        number_key(maximum),
    )


def get_nested(data: dict[str, Any], path: list[str]) -> Any:
    current: Any = data
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return None
        current = current[key]
    return current


def find_first(data: dict[str, Any], paths: list[list[str]]) -> Any:
    for path in paths:
        value = get_nested(data, path)
        if value not in (None, ''):
            return value
    return None


def get_json_values(data: dict[str, Any]) -> dict[str, Any]:
    return {
        'start': find_first(data, [
            ['properties', 'start_datetime'],
            ['properties', 'datetime'],
            ['start_datetime'],
            ['datetime'],
            ['storm_date'],
            ['start_date'],
        ]),
        'mean': find_first(data, [
            ['properties', 'aorc:statistics', 'mean'],
            ['properties', 'statistics', 'mean'],
            ['aorc:statistics', 'mean'],
            ['statistics', 'mean'],
            ['mean'],
        ]),
        'min': find_first(data, [
            ['properties', 'aorc:statistics', 'min'],
            ['properties', 'statistics', 'min'],
            ['aorc:statistics', 'min'],
            ['statistics', 'min'],
            ['min'],
        ]),
        'max': find_first(data, [
            ['properties', 'aorc:statistics', 'max'],
            ['properties', 'statistics', 'max'],
            ['aorc:statistics', 'max'],
            ['statistics', 'max'],
            ['max'],
        ]),
    }


def compare_date(csv_value: Any, json_value: Any) -> FieldCheck:
    csv_datetime = parse_datetime(csv_value)
    json_datetime = parse_datetime(json_value)
    difference = ''
    if csv_datetime is not None and json_datetime is not None:
        delta_hours = (json_datetime - csv_datetime).total_seconds() / 3600
        difference = f'{delta_hours:.6f}'.rstrip('0').rstrip('.')
    return FieldCheck(
        name='start_date',
        csv_value='' if csv_value is None else str(csv_value),
        json_value='' if json_value is None else str(json_value),
        csv_normalized=format_datetime(csv_datetime),
        json_normalized=format_datetime(json_datetime),
        difference=difference,
        match=(
            csv_datetime is not None
            and json_datetime is not None
            and csv_datetime == json_datetime
        ),
    )


def compare_number(name: str, csv_value: Any, json_value: Any) -> FieldCheck:
    csv_number = parse_number(csv_value)
    json_number = parse_number(json_value)
    difference = ''
    match = False
    if csv_number is not None and json_number is not None:
        delta = json_number - csv_number
        difference = f'{delta:.10g}'
        match = abs(delta) <= NUMERIC_TOLERANCE
    return FieldCheck(
        name=name,
        csv_value='' if csv_value is None else str(csv_value),
        json_value='' if json_value is None else str(json_value),
        csv_normalized='' if csv_number is None else f'{csv_number:.10g}',
        json_normalized='' if json_number is None else f'{json_number:.10g}',
        difference=difference,
        match=match,
    )


def find_json_file(folder: Path) -> tuple[Path | None, str]:
    expected = folder / f'{folder.name}.json'
    if expected.is_file():
        return expected, ''
    json_files = sorted(folder.glob('*.json'))
    if len(json_files) == 1:
        return json_files[0], f'Used the only JSON file in the folder: {json_files[0].name}'
    if not json_files:
        return None, 'No JSON file found in the rank folder.'
    return json_files[0], f'Multiple JSON files found; used {json_files[0].name}.'


def join_ranks(ranks: list[int]) -> str:
    return '|'.join(str(rank) for rank in ranks)


def join_rank_offsets(current_rank: int, ranks: list[int]) -> str:
    return '|'.join(str(rank - current_rank) for rank in ranks)


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

## Input checks

In [ ]:
csv_df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
csv_df.columns = [clean_header(column) for column in csv_df.columns]

required_columns = {'storm_date', 'mean', 'min', 'max'}
missing_columns = sorted(required_columns - set(csv_df.columns))
if missing_columns:
    raise ValueError(f'Missing ranked CSV columns: {missing_columns}')

def rank_for_row(row: pd.Series, row_number: int) -> int:
    for column in ('por_rank', 'rank', 'storm_rank'):
        value = row.get(column, '')
        if value not in (None, ''):
            try:
                return int(float(value))
            except (TypeError, ValueError):
                pass
    return row_number


csv_df['audit_rank'] = [
    rank_for_row(row, row_number)
    for row_number, (_, row) in enumerate(csv_df.iterrows(), start=1)
]
duplicate_ranks = csv_df.loc[csv_df['audit_rank'].duplicated(keep=False), 'audit_rank'].tolist()
if duplicate_ranks:
    raise ValueError(f'Duplicate ranks in the CSV: {sorted(set(duplicate_ranks))}')

csv_by_rank = {int(row['audit_rank']): row for _, row in csv_df.iterrows()}
rank_folders = sorted(
    [path for path in EVENTS_DIR.iterdir() if path.is_dir() and path.name.isdigit()],
    key=lambda path: int(path.name),
)
folder_ranks = {int(folder.name) for folder in rank_folders}
csv_ranks = set(csv_by_rank)

csv_missing_folders_df = csv_df[csv_df['audit_rank'].isin(sorted(csv_ranks - folder_ranks))].copy()
folders_missing_csv = sorted(folder_ranks - csv_ranks)

print(f'Ranked CSV rows: {len(csv_df):,}')
print(f'Numbered event folders: {len(rank_folders):,}')
print(f'CSV rows without folders: {len(csv_missing_folders_df):,}')
print(f'Folders without CSV rows: {len(folders_missing_csv):,}')

## Compare ranked rows with event JSON files

In [ ]:
DETAIL_COLUMNS = [
    'rank', 'review_status', 'match_score', 'fields_mismatched', 'mismatch_count',
    'csv_storm_date', 'json_start_datetime', 'csv_start_normalized',
    'json_start_normalized', 'start_date_delta_hours', 'start_date_match',
    'csv_mean', 'json_mean', 'mean_diff_json_minus_csv', 'mean_match',
    'csv_min', 'json_min', 'min_diff_json_minus_csv', 'min_match',
    'csv_max', 'json_max', 'max_diff_json_minus_csv', 'max_match',
    'json_start_found_csv_ranks', 'json_start_rank_offsets',
    'json_full_values_found_csv_ranks', 'json_full_values_rank_offsets',
    'json_path', 'notes',
]


def blank_result(rank: int, status: str, csv_row: pd.Series | None, json_path: Path | None,
                 mismatch_field: str, notes: str) -> dict[str, Any]:
    csv_value = (lambda key: '' if csv_row is None else str(csv_row.get(key, '')))
    return {
        'rank': rank,
        'review_status': status,
        'match_score': '0/4',
        'fields_mismatched': mismatch_field,
        'mismatch_count': 1,
        'csv_storm_date': csv_value('storm_date'),
        'json_start_datetime': '',
        'csv_start_normalized': format_datetime(parse_datetime(csv_value('storm_date'))),
        'json_start_normalized': '',
        'start_date_delta_hours': '',
        'start_date_match': False,
        'csv_mean': csv_value('mean'),
        'json_mean': '',
        'mean_diff_json_minus_csv': '',
        'mean_match': False,
        'csv_min': csv_value('min'),
        'json_min': '',
        'min_diff_json_minus_csv': '',
        'min_match': False,
        'csv_max': csv_value('max'),
        'json_max': '',
        'max_diff_json_minus_csv': '',
        'max_match': False,
        'json_start_found_csv_ranks': '',
        'json_start_rank_offsets': '',
        'json_full_values_found_csv_ranks': '',
        'json_full_values_rank_offsets': '',
        'json_path': '' if json_path is None else str(json_path),
        'notes': notes,
    }


csv_start_rank_map: dict[str, list[int]] = {}
csv_full_value_rank_map: dict[tuple[str, str, str, str], list[int]] = {}
for rank, row in csv_by_rank.items():
    start_key = format_datetime(parse_datetime(row.get('storm_date', '')))
    full_key = storm_value_key(
        row.get('storm_date', ''), row.get('mean', ''),
        row.get('min', ''), row.get('max', ''),
    )
    csv_start_rank_map.setdefault(start_key, []).append(rank)
    csv_full_value_rank_map.setdefault(full_key, []).append(rank)

detail_rows: list[dict[str, Any]] = []
for folder in rank_folders:
    rank = int(folder.name)
    csv_row = csv_by_rank.get(rank)
    json_path, json_note = find_json_file(folder)

    if csv_row is None:
        detail_rows.append(blank_result(
            rank, 'NO_CSV_ROW', None, json_path, 'csv_row',
            'No CSV row found for this numbered folder.',
        ))
        continue

    if json_path is None:
        detail_rows.append(blank_result(
            rank, 'NO_JSON', csv_row, None, 'json_file', json_note,
        ))
        continue

    try:
        with json_path.open('r', encoding='utf-8') as handle:
            json_data = json.load(handle)
        json_values = get_json_values(json_data)
    except Exception as error:
        note = f'{json_note}; ' if json_note else ''
        note += f'JSON read error: {error}'
        detail_rows.append(blank_result(
            rank, 'JSON_READ_ERROR', csv_row, json_path, 'json_read', note,
        ))
        continue

    checks = [
        compare_date(csv_row.get('storm_date', ''), json_values['start']),
        compare_number('mean', csv_row.get('mean', ''), json_values['mean']),
        compare_number('min', csv_row.get('min', ''), json_values['min']),
        compare_number('max', csv_row.get('max', ''), json_values['max']),
    ]
    mismatched_fields = [check.name for check in checks if not check.match]
    status = 'MATCH' if not mismatched_fields else 'DIFF'
    json_start_ranks = csv_start_rank_map.get(checks[0].json_normalized, [])
    json_full_key = storm_value_key(
        checks[0].json_normalized,
        checks[1].json_normalized,
        checks[2].json_normalized,
        checks[3].json_normalized,
    )
    json_full_ranks = csv_full_value_rank_map.get(json_full_key, [])

    detail_rows.append({
        'rank': rank,
        'review_status': status,
        'match_score': f'{4 - len(mismatched_fields)}/4',
        'fields_mismatched': '|'.join(mismatched_fields),
        'mismatch_count': len(mismatched_fields),
        'csv_storm_date': checks[0].csv_value,
        'json_start_datetime': checks[0].json_value,
        'csv_start_normalized': checks[0].csv_normalized,
        'json_start_normalized': checks[0].json_normalized,
        'start_date_delta_hours': checks[0].difference,
        'start_date_match': checks[0].match,
        'csv_mean': checks[1].csv_value,
        'json_mean': checks[1].json_value,
        'mean_diff_json_minus_csv': checks[1].difference,
        'mean_match': checks[1].match,
        'csv_min': checks[2].csv_value,
        'json_min': checks[2].json_value,
        'min_diff_json_minus_csv': checks[2].difference,
        'min_match': checks[2].match,
        'csv_max': checks[3].csv_value,
        'json_max': checks[3].json_value,
        'max_diff_json_minus_csv': checks[3].difference,
        'max_match': checks[3].match,
        'json_start_found_csv_ranks': join_ranks(json_start_ranks),
        'json_start_rank_offsets': join_rank_offsets(rank, json_start_ranks),
        'json_full_values_found_csv_ranks': join_ranks(json_full_ranks),
        'json_full_values_rank_offsets': join_rank_offsets(rank, json_full_ranks),
        'json_path': str(json_path),
        'notes': json_note,
    })

detail_df = pd.DataFrame(detail_rows, columns=DETAIL_COLUMNS).sort_values('rank').reset_index(drop=True)
mismatch_df = detail_df[detail_df['review_status'] != 'MATCH'].copy()
print(f'Compared {len(detail_df):,} numbered folders.')

## Results summary

In [ ]:
status_counts = detail_df['review_status'].value_counts().to_dict()
field_counts = (
    mismatch_df['fields_mismatched']
    .str.split('|')
    .explode()
    .replace('', pd.NA)
    .dropna()
    .value_counts()
)
field_summary_df = field_counts.rename_axis('Field').reset_index(name='Mismatch count')

summary_rows = [
    ('Run time UTC', datetime.now(timezone.utc).isoformat()),
    ('Events folder', str(EVENTS_DIR.resolve())),
    ('Ranked CSV', str(CSV_PATH.resolve())),
    ('Ranked CSV SHA-256', file_sha256(CSV_PATH)),
    ('Numeric tolerance', NUMERIC_TOLERANCE),
    ('Ranked CSV rows', len(csv_df)),
    ('Numbered event folders', len(rank_folders)),
    ('Matching folders', status_counts.get('MATCH', 0)),
    ('Folders with differences', status_counts.get('DIFF', 0)),
    ('Folders missing JSON', status_counts.get('NO_JSON', 0)),
    ('Folders with JSON read errors', status_counts.get('JSON_READ_ERROR', 0)),
    ('Folders without CSV rows', status_counts.get('NO_CSV_ROW', 0)),
    ('CSV rows without folders', len(csv_missing_folders_df)),
]
summary_df = pd.DataFrame(summary_rows, columns=['Metric', 'Value'])

display(summary_df)
if field_summary_df.empty:
    print('No field mismatches were found.')
else:
    display(field_summary_df)

review_item_count = len(mismatch_df) + len(csv_missing_folders_df)
if review_item_count == 0:
    print('Result: every ranked CSV row has a matching event folder and JSON file.')
else:
    print(f'Result: {review_item_count:,} items need review.')
    if not mismatch_df.empty:
        print(f'Folder or JSON differences: {len(mismatch_df):,}. Showing the first {DISPLAY_ROWS}.')
        display(mismatch_df.head(DISPLAY_ROWS))
    if not csv_missing_folders_df.empty:
        print(f'CSV rows without numbered folders: {len(csv_missing_folders_df):,}.')
        display(csv_missing_folders_df.head(DISPLAY_ROWS))

## Write the Excel report

The workbook uses red cells for differences, orange cells for missing or unreadable files, and green cells for matching folders.

In [ ]:
missing_folder_columns = list(csv_df.columns)
if csv_missing_folders_df.empty:
    csv_missing_folders_df = pd.DataFrame(columns=missing_folder_columns)

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    field_summary_df.to_excel(writer, sheet_name='Field Summary', index=False)
    mismatch_df.to_excel(writer, sheet_name='Mismatches', index=False)
    detail_df.to_excel(writer, sheet_name='All Comparisons', index=False)
    csv_missing_folders_df.to_excel(writer, sheet_name='CSV Missing Folders', index=False)

    workbook = writer.book
    header_fill = PatternFill('solid', fgColor='37474F')
    header_font = Font(color='FFFFFF', bold=True)
    match_fill = PatternFill('solid', fgColor='C8E6C9')
    diff_fill = PatternFill('solid', fgColor='FFCDD2')
    warning_fill = PatternFill('solid', fgColor='FFE0B2')

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = 'A2'
        worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')

        for column_index, column_cells in enumerate(worksheet.columns, start=1):
            values = [str(cell.value) if cell.value is not None else '' for cell in column_cells]
            width = min(max(max((len(value) for value in values), default=0) + 2, 10), 55)
            worksheet.column_dimensions[get_column_letter(column_index)].width = width

        headers = {cell.value: cell.column for cell in worksheet[1]}
        status_column = headers.get('review_status')
        if status_column:
            for row_number in range(2, worksheet.max_row + 1):
                status_cell = worksheet.cell(row=row_number, column=status_column)
                if status_cell.value == 'MATCH':
                    status_cell.fill = match_fill
                elif status_cell.value == 'DIFF':
                    status_cell.fill = diff_fill
                else:
                    status_cell.fill = warning_fill

        for flag_name in ('start_date_match', 'mean_match', 'min_match', 'max_match'):
            flag_column = headers.get(flag_name)
            if not flag_column:
                continue
            for row_number in range(2, worksheet.max_row + 1):
                flag_cell = worksheet.cell(row=row_number, column=flag_column)
                if flag_cell.value is True:
                    flag_cell.fill = match_fill
                elif flag_cell.value is False:
                    flag_cell.fill = diff_fill

        if worksheet.title == 'CSV Missing Folders':
            for row_number in range(2, worksheet.max_row + 1):
                for column_number in range(1, worksheet.max_column + 1):
                    worksheet.cell(row=row_number, column=column_number).fill = warning_fill

        if worksheet.title == 'Summary':
            problem_metrics = {
                'Folders with differences', 'Folders missing JSON',
                'Folders with JSON read errors', 'Folders without CSV rows',
                'CSV rows without folders',
            }
            for row_number in range(2, worksheet.max_row + 1):
                metric = worksheet.cell(row=row_number, column=1).value
                value_cell = worksheet.cell(row=row_number, column=2)
                if metric == 'Matching folders':
                    value_cell.fill = match_fill
                elif metric in problem_metrics and isinstance(value_cell.value, (int, float)):
                    value_cell.fill = warning_fill if value_cell.value else match_fill

print('Excel report written:', EXCEL_PATH)
print(f'Matching folders: {status_counts.get("MATCH", 0):,}')
print(f'Items needing review: {review_item_count:,}')